```bash
pwd


pwd <specific directory>
```

prints current working directory


```bash
ls

ls <specific directory>

ls -a  # for showing hidden files and directory
```

list contents


```bash
cd  #  "cd" goes to home directory

cd ~ # also goes to home dirctory

cd / # goes to the root directory

cd .. # goes one folder backward

cd ../.. # goes two folder backward

```


```bash
mkdir ~/dev # create new folder

mkdir ~/dev/devops # create nested folder

mkdir -p ~/dev/devops # if dev folder not exists  it create both dev folder and devops folder

```


# 🐧 Level 1 – Linux Survival Lab (for WSL Ubuntu)

> **Read this first:**  
> Open your WSL Ubuntu terminal. You’ll **type every command yourself** and compare what you see with the outputs shown here.  
> When you see `$`, it means the shell prompt – do **not** type the `$`.


## ⚙️ 0. Lab Setup (install missing tools)

Some commands (like `tree`) aren’t installed by default. Run this once:

```bash
sudo apt update && sudo apt install -y tree
```

Now everything in this lab will work.


## 📂 1. Navigating the Filesystem (Hands‑On)

### Step 1.1 – Where am I?

```bash
pwd
```

Expected output:

```
/home/your_username
```

(Your username will be different – that’s fine.)

### Step 1.2 – List files with details

```bash
ls -lah
```

You’ll see all files (including hidden) with human‑readable sizes, like:

```
drwxr-xr-x 2 yourname yourname 4.0K Jun  8 10:00 Documents
-rw------- 1 yourname yourname 1.2K Jun  8 09:45 .bashrc
```

### Step 1.3 – Move to a system directory and verify

```bash
cd /var/log
pwd
```

Output:

```
/var/log
```

### Step 1.4 – Jump back to where you were before (huge time‑saver)

```bash
cd -
pwd
```

Now you’re back in your home directory. `cd -` toggles between the last two locations.

### Step 1.5 – Look at a directory as a tree

```bash
cd /var/log
tree -L 2 -d   # show only directories, depth 2
```

You’ll see a tidy tree of `/var/log` subdirectories.  
If you skip `-L 2`, `tree` would show **all** files recursively and could flood the terminal.

### Step 1.6 – Find recently changed files

```bash
find /tmp -mmin -5 -ls 2>/dev/null
```

This lists every file in `/tmp` modified in the last 5 minutes. The `2>/dev/null` silences permission‑denied noise.


### 🧠 What you just learned (and will use daily)

- `pwd` = print working directory
- `cd -` = toggle to previous directory
- `ls -lah` = see everything human‑readable
- `tree -L 2` = visual directory map (install it if you haven’t)

**Try it yourself:**  
Create a nested path `/tmp/labtest/alpha/beta` using only `mkdir -p`, then use `tree` to see it.

<details>
<summary>Solution</summary>

```bash
mkdir -p /tmp/labtest/alpha/beta
tree /tmp/labtest
```

</details>


## 📖 2. Reading Files (Hands‑On)

We’ll create a sample log file to work with.

```bash
# Create a 15‑line file with timestamps
for i in {1..15}; do echo "[$(date +%H:%M:%S)] INFO Log entry $i"; done > /tmp/sample.log
```

### Step 2.1 – Peek at the top and bottom

```bash
head -n 3 /tmp/sample.log
```

Output:

```
[14:02:10] INFO Log entry 1
[14:02:10] INFO Log entry 2
[14:02:10] INFO Log entry 3
```

```bash
tail -n 4 /tmp/sample.log
```

You see the last 4 lines.

### Step 2.2 – Watch a file in real time (like `tail -f` but we’ll simulate)

Open a **second terminal** (or just imagine):

```bash
# In terminal 1:
tail -f /tmp/sample.log
```

Now, in terminal 2:

```bash
echo "NEW LINE ADDED" >> /tmp/sample.log
```

Switch back to terminal 1 – the line appears instantly. Press `Ctrl+C` to stop.

### Step 2.3 – Read a large file safely with less

```bash
less /var/log/syslog   # press 'q' to quit
```

Inside `less`:

- `/error` – search for “error”
- `n` – next match, `N` – previous match
- `G` – jump to end, `g` – jump to start

### Step 2.4 – Display the file in reverse (most recent first)

```bash
tac /tmp/sample.log | head -5
```

You’ll see the newest line first – great for checking the tail of a log without scrolling.


### 🧠 Key wisdom

- **Never `cat` a huge file** – use `less` or `tail`.
- `tail -F` (capital F) survives log rotation; `-f` doesn’t.
- `tac` reverses lines – useful for logs that grow at the end.

**Try it yourself:** Create a 200‑line file with `seq 200 > /tmp/bigfile.txt` and practice paging through it with `less`.


## 🛠️ 3. Creating & Destroying Files/Directories (Hands‑On)

### Step 3.1 – Create a complex directory tree with one command

```bash
mkdir -p ~/lab/{backup,current/{config,logs},archive}
```

Verify with:

```bash
tree ~/lab
```

Output:

```
/home/yourname/lab
├── archive
├── backup
└── current
    ├── config
    └── logs
```

### Step 3.2 – Create files, copy, and rename

```bash
cd ~/lab/current/config
touch app.conf            # empty file
cp app.conf app.conf.bak  # backup
mv app.conf.bak app.conf.old
ls -l
```

You’ll see the renamed file.

### Step 3.3 – Create a symbolic link

```bash
cd ~/lab
ln -s current/releases/v1.0 current_live   # let's first create the release dir
mkdir -p current/releases/v1.0
ln -s current/releases/v1.0 current_live
ls -l current_live
```

Output shows the link target:

```
lrwxrwxrwx ... current_live -> current/releases/v1.0
```

### Step 3.4 – Delete safely

```bash
rm -ri ~/lab/archive   # interactive recursive – asks for each file
```

(If `archive` is empty, it just asks to remove the directory – answer `y`.)

**Important:** Without `-i`, `rm -rf` is instant and permanent. Always double‑check your path.


### 🧠 Best practices

- Use `cp -p` to preserve permissions/timestamps (`cp -p source dest`).
- Use `mv -i` to avoid accidentally overwriting.
- Symbolic links enable zero‑downtime deployments – you just repoint the link.

**Try it yourself:** Create a directory `~/lab/trash`, put a few files in it, then delete it with `rm -rfi` to practice interactive deletion.


## 🔐 4. Basic Permissions (Hands‑On)

### Step 4.1 – Understand the listing

```bash
cd ~
touch secret.txt
ls -l secret.txt
```

Output like:

```
-rw-r--r-- 1 you you 0 Jun  8 14:10 secret.txt
```

Interpretation: `-rw-r--r--` means owner can read/write, group can read, others can read.

### Step 4.2 – Restrict a file to owner only

```bash
chmod 600 secret.txt
ls -l secret.txt
```

Now: `-rw-------` – only the owner can read/write.

### Step 4.3 – Make a script executable

```bash
echo '#!/bin/bash' > myscript.sh
echo 'echo "Hello from script"' >> myscript.sh
chmod +x myscript.sh
./myscript.sh
```

Output: `Hello from script`

### Step 4.4 – View permissions in octal (for scripting)

```bash
stat -c '%a %n' secret.txt
```

Output: `600 secret.txt`

### Step 4.5 – Set a secure default `umask`

```bash
umask 0077
touch newfile
ls -l newfile
```

The file will be created with `-rw-------` (because `umask 0077` strips all group/other permissions).


### 🧠 Permission octal cheat sheet

| Octal | Symbolic | Meaning              |
| ----- | -------- | -------------------- |
| 7     | rwx      | read, write, execute |
| 6     | rw-      | read, write          |
| 5     | r-x      | read, execute        |
| 4     | r--      | read only            |
| 0     | ---      | no permissions       |

Position: `owner group others`. So `755` = owner rwx, group r‑x, others r‑x.

**Try it yourself:** Create a directory `~/lab/data`, set its permissions to `700`, and verify that another user (if you have one) cannot list it.


## ❓ 5. Getting Help Without Google (Hands‑On)

### Step 5.1 – Built‑in help for a command

```bash
ls --help | head -20
```

Shows common flags quickly.

### Step 5.2 – One‑line description

```bash
whatis ls
```

Output: `ls (1) - list directory contents`

### Step 5.3 – Full manual page

```bash
man grep
```

Press `/` and type `-i` to find the explanation for the `-i` flag. Press `q` to exit.

### Step 5.4 – Search manpages by topic

```bash
man -k disk
```

Lists all commands related to “disk”, like `df`, `du`, `fdisk`.

### Step 5.5 – Help on bash built‑ins

```bash
help for
```

(Works only for shell built‑ins, not external commands.)


## 💻 6. Your First Bash Scripts (Hands‑On)

### Step 6.1 – Write and run a simple script

```bash
cat > ~/lab/mystats.sh << 'EOF'
#!/bin/bash
echo "==== System Stats ===="
echo "Date: $(date)"
echo "Uptime: $(uptime -p)"
echo "Disk usage:"
df -h / | tail -1
EOF
chmod +x ~/lab/mystats.sh
./mystats.sh
```

You’ll see a clean system summary.

### Step 6.2 – A script with if/else

```bash
cat > ~/lab/check_log.sh << 'EOF'
#!/bin/bash
LOG="/var/log/syslog"
if [ -f "$LOG" ]; then
    echo "Log exists, last 3 lines:"
    tail -n 3 "$LOG"
else
    echo "Log not found!"
    exit 1
fi
EOF
chmod +x ~/lab/check_log.sh
./check_log.sh
```

### Step 6.3 – Loop through files

```bash
cat > ~/lab/count_files.sh << 'EOF'
#!/bin/bash
for file in /etc/*.conf; do
    if [ -f "$file" ]; then
        echo "$file has $(wc -l < "$file") lines"
    fi
done
EOF
chmod +x ~/lab/count_files.sh
./count_files.sh | head -10
```


### 🧠 Scripting safety rules

- Always `"quote"` your variables (`"$file"`, not `$file`).
- Use `$(command)` instead of backticks.
- Inside `[ ]` you **must** have spaces: `[ -f "$f" ]` ✅, `[-f"$f"]` ❌.
- Make scripts executable with `chmod +x scriptname`.

**Try it yourself:** Modify `mystats.sh` to also show memory usage (`free -h`).


## 🚀 Daily Aliases (add to `~/.bashrc`)

Run these now, then add them permanently later:

```bash
alias ll='ls -lah'
alias ..='cd ..'
alias ...='cd ../..'
alias rm='rm -i'
alias cp='cp -i'
alias mv='mv -i'
alias grep='grep --color=auto'
```

Activate them:

```bash
source ~/.bashrc
```


## 🧪 Final Challenge – Tie It All Together

Create a mini “deployment” simulation entirely in your terminal:

1. Create directory structure:
   ```bash
   mkdir -p ~/deploy/app/{releases/2026-06-08,current}
   ```
2. Create a dummy config and set permissions:
   ```bash
   echo "DB_PASS=secret" > ~/deploy/app/config.env
   chmod 600 ~/deploy/app/config.env
   ```
3. Write a script `deploy.sh` that:
   - Checks if `config.env` exists (and readable).
   - Copies everything from `releases/2026-06-08` to `current` (simulate with `cp -r`).
   - Prints “Deployment successful” or an error.
4. Run it and verify the symlink logic:
   ```bash
   # Actually use a symbolic link for zero-downtime:
   rm -rf ~/deploy/app/current   # be careful, we know it's empty
   ln -s ~/deploy/app/releases/2026-06-08 ~/deploy/app/current
   ls -l ~/deploy/app/current
   ```

<details>
<summary>Sample deploy.sh</summary>

```bash
#!/bin/bash
APP_BASE=~/deploy/app
RELEASE="$APP_BASE/releases/2026-06-08"
CURRENT="$APP_BASE/current"
CONFIG="$APP_BASE/config.env"

if [ ! -f "$CONFIG" ]; then
    echo "Missing config!" >&2
    exit 1
fi

echo "Deploying release..."
# (In real life you'd extract/copy files here)
ln -sfn "$RELEASE" "$CURRENT" 2>/dev/null
echo "Deployment successful. Current -> $(readlink -f "$CURRENT")"
```

Make it executable and run it.

</details>
